# Preclass
AI for Materials Science — Hands-on session 5: Train and Test ML/DL Models

This is a warm-up for the two tools you will use in class: scikit-learn for classical machine
learning and PyTorch for a small neural network. It takes about 30 minutes. There is nothing to submit.

All data in this notebook are synthetic, generated with NumPy as y = f(x) + noise. No external dataset is used. The in-class session uses real battery electrode data.

## What this notebook covers

### 0 · Setup
Check the libraries and download the course material.

### A · scikit-learn: fit, predict, score
Make a dataset, split it, fit a line, and read three error metrics.

### B · PyTorch: tensors, gradients, a training loop
See what a tensor is, let PyTorch compute a gradient, and train a one-neuron model in a ten-line loop.

---

**First:** File ▸ Save a copy in Drive, and work in your copy.

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

**Shared or lab PC?** Sign out of Google when you finish and choose Runtime ▸ Disconnect and delete runtime.

## 0. Setup

### 0-1. Check the libraries
Everything this notebook needs ships with Colab: NumPy, pandas, scikit-learn, Matplotlib and PyTorch.
There is nothing to install; this cell only prints the versions you are running.

In [ ]:
import numpy, pandas, sklearn, matplotlib, torch
modules = [("numpy", numpy), ("pandas", pandas), ("scikit-learn", sklearn),
           ("matplotlib", matplotlib), ("torch", torch)]
for name, module in modules:
    print(f"{name:13s} {module.__version__}")

### 0-2. Download the course material
The same repository as in the earlier sessions. This notebook does not read any file from it, but the
in-class notebook will, so it is worth seeing that the clone works.

In [ ]:
## A leading ! runs a terminal command instead of Python.
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

### 0-4. Set the file paths

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")      ## course data (used in class)
OUTPUT_DIR = Path("outputs")             ## figures you make today
OUTPUT_DIR.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. scikit-learn: fit, predict, score

Every supervised model in this course follows the same four steps: **split** the data, **fit** a
model on the training part, **predict** on the test part, and **score** the predictions. We practise
the steps on a dataset whose true rule we know, so you can check what the model learned.

### A-1. Make a synthetic dataset
200 points on the line y = 2.5 x + 1 with Gaussian noise (standard deviation 1.5).

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, size=200)
y = 2.5 * x + 1.0 + rng.normal(0, 1.5, size=200)   ## true slope 2.5, intercept 1.0
data = pd.DataFrame({"x": x, "y": y})
data.head()

### A-2. Split into training and test sets
The model never sees the test rows while it is being fitted. That is what makes the test score an
honest estimate of how it will do on new data.

In [ ]:
X = data[["x"]]            ## features must be a 2-D table, even with one column
y = data["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=0)
print("train rows:", len(X_train), " test rows:", len(X_test))

### A-3. Fit a linear model
Compare the fitted slope and intercept with the true values 2.5 and 1.0.

In [ ]:
model = LinearRegression().fit(X_train, y_train)
print(f"slope     {model.coef_[0]:.3f}   (true 2.5)")
print(f"intercept {model.intercept_:.3f}   (true 1.0)")

### A-4. Three metrics
- **MAE** (mean absolute error): the typical size of a miss, in the units of y.
- **RMSE** (root mean squared error): like MAE but large misses count more.
- **R²**: the share of the variance in y that the model explains (1.0 is perfect, 0.0 is "predict the mean").

In [ ]:
def scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

table = pd.DataFrame({"train": scores(y_train, model.predict(X_train)),
                      "test": scores(y_test, model.predict(X_test))})
table.round(3)

### A-5. Parity plot
A parity plot puts the true value on the x-axis and the prediction on the y-axis. Perfect predictions
sit on the diagonal; the spread around it is the error you just measured.

In [ ]:
y_pred = model.predict(X_test)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(y_test, y_pred, s=18)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("true y"); ax.set_ylabel("predicted y"); ax.set_title("Test set parity")
fig.savefig(OUTPUT_DIR / "preclass_parity.png", dpi=150, bbox_inches="tight")
plt.show()

## B. PyTorch: tensors, gradients, a training loop

A neural network is fitted by **gradient descent**: compute how wrong the model is, ask PyTorch for the
gradient of that error with respect to every parameter, and take a small step downhill. Repeat.

### B-1. Tensors
A tensor is PyTorch's array. It converts to and from NumPy, and it remembers which device and dtype it uses.

In [ ]:
t = torch.tensor(data["x"].to_numpy(), dtype=torch.float32)
print("shape", tuple(t.shape), " dtype", t.dtype, " first three", t[:3])
print("back to NumPy:", t.numpy()[:3])

### B-2. Automatic gradients
With `requires_grad=True`, PyTorch records every operation on `w`. After `backward()` the derivative of
the loss with respect to `w` is in `w.grad`. Here loss = (3w − 9)², so d(loss)/dw = 2(3w − 9)·3 = −18 at w = 2.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (3 * w - 9) ** 2
loss.backward()
print("loss", loss.item(), " gradient", w.grad.item(), " (expected -18.0)")

### B-3. A training loop in ten lines
`nn.Linear(1, 1)` is a one-neuron model: y = w·x + b. The loop below is the same loop we will use for the
multi-layer network in class; only the model gets bigger.

In [ ]:
X_t = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
torch.manual_seed(0)
net = nn.Linear(1, 1)
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)
loss_fn = nn.MSELoss()
losses = []
for epoch in range(200):
    optimizer.zero_grad()              ## 1. forget the previous gradients
    loss = loss_fn(net(X_t), y_t)      ## 2. how wrong are we?
    loss.backward()                    ## 3. gradient w.r.t. w and b
    optimizer.step()                   ## 4. one step downhill
    losses.append(loss.item())
print(f"weight {net.weight.item():.3f} (true 2.5)",
      f"bias {net.bias.item():.3f} (true 1.0)")

### B-4. Watch the loss fall
The loss should drop quickly and flatten out. The assert at the end is the one-line check that training
actually moved in the right direction.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(losses)
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss")
fig.savefig(OUTPUT_DIR / "preclass_loss.png", dpi=150, bbox_inches="tight")
plt.show()
assert losses[-1] < losses[0], "training did not reduce the loss"

X_te = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
with torch.no_grad():
    resid = net(X_te).numpy().ravel() - y_test.to_numpy()
print(f"test RMSE  PyTorch {np.sqrt(np.mean(resid ** 2)):.3f}",
      f"  scikit-learn {table.loc['RMSE', 'test']:.3f}")

## Data sources
All data in this notebook are synthetic, generated with NumPy as y = f(x) + noise. No external dataset is used. The course repository is cloned only so that the in-class notebook can read its `Data/` folder.